# 🧹 Notebook 1: Preprocessing — Deteksi Bahasa + Translate

**Input** : `/kaggle/input/datasethotel/DatasetHotel/` (raw CSV)

**Output** : `/kaggle/working/DatasetHotelCLEAN/` (cleaned CSV)

**Alur:**
1. Deteksi bahasa setiap review
2. Jika sudah Bahasa Indonesia → simpan langsung (skip translate)
3. Jika Bahasa Inggris/lain → translate ke Indonesia (NLLB-200)
4. Normalisasi slang & teks

> Setelah selesai, download output lalu upload sebagai Kaggle Dataset baru
> untuk dipakai di **Notebook 2 (Model)**.

In [ ]:
# CELL 1: INSTALASI
!pip install -q langdetect
!pip install -q transformers sentencepiece sacremoses
print('Instalasi selesai.')

In [ ]:
# CELL 2: IMPORT & CEK GPU
import os, re, gc, glob
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42  # reproducible

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device.upper()}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
    # CELL 3: KONFIGURASI PATH
    # Input: dataset Kaggle yang sudah ditambahkan
    INPUT_ROOT  = '/kaggle/input/datasethotel/DatasetHotel'
    OUTPUT_ROOT = '/kaggle/working/DatasetHotelCLEAN'

    # Folder yang akan diproses (sesuai struktur dataset)
    FOLDERS = ['BUMNB3', 'BUMNB4', 'BUMNB5', 'KOMPETITORB3', 'KOMPETITORB4', 'KOMPETITORB5']

    # Buat folder output
    for folder in FOLDERS:
        os.makedirs(os.path.join(OUTPUT_ROOT, folder), exist_ok=True)

    # Hitung total file
    all_files = glob.glob(os.path.join(INPUT_ROOT, '**', '*.csv'), recursive=True)
    print(f'Total file CSV ditemukan: {len(all_files)}')
    for f in all_files[:5]:
        print(' ', f)

In [ ]:
# CELL 4: LOAD MODEL TRANSLATE (NLLB-200)
# T4 punya 15GB VRAM — bisa pakai float16 penuh
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
print(f'Loading {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
translate_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32
).to(device)

print('Model translate siap!')

In [ ]:
# CELL 5: KAMUS SLANG & FUNGSI-FUNGSI

SLANG_DICT = {
    'yg': 'yang', 'ga': 'tidak', 'gak': 'tidak', 'nggak': 'tidak',
    'tp': 'tapi', 'krn': 'karena', 'utk': 'untuk', 'sdh': 'sudah',
    'udh': 'sudah', 'udah': 'sudah', 'blm': 'belum', 'dgn': 'dengan',
    'dlm': 'dalam', 'bgt': 'banget', 'tdk': 'tidak', 'jgn': 'jangan',
    'krg': 'kurang', 'sy': 'saya', 'ak': 'aku', 'kalo': 'kalau',
    'kl': 'kalau', 'dr': 'dari', 'bs': 'bisa', 'tmn': 'teman',
    'bgs': 'bagus', 'dtg': 'datang', 'br': 'baru', 'ok': 'oke',
    'thx': 'terima kasih', 'makasih': 'terima kasih', 'tks': 'terima kasih',
    'chekout': 'check out', 'chekin': 'check in', 'pd': 'pada',
    'pake': 'pakai', 'sm': 'sama', 'lbh': 'lebih', 'bkn': 'bukan',
    'spt': 'seperti', 'jd': 'jadi', 'aja': 'saja', 'aj': 'saja',
    'kmr': 'kamar', 'kmar': 'kamar', 'dl': 'dulu', 'skrg': 'sekarang',
    'dg': 'dengan', 'emg': 'memang', 'emang': 'memang', 'hrs': 'harus',
    'mgkn': 'mungkin', 'knp': 'kenapa', 'gmn': 'gimana', 'gitu': 'begitu',
    'ngga': 'tidak', 'bener': 'benar', 'nih': '', 'sih': '', 'deh': '', 'dong': '',
}

# Bahasa yang TIDAK perlu ditranslate
SKIP_LANGS = {'id', 'ms'}  # Indonesia & Melayu


def detect_lang(text: str) -> str:
    """Deteksi bahasa. Default 'id' jika gagal."""
    try:
        return detect(str(text))
    except Exception:
        return 'id'


def clean_text(text: str) -> str:
    """Normalisasi teks: lowercase, hapus pengulangan, normalisasi slang."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'(.)\1{2,}', r'\1', text)   # 'bagussss' → 'bagus'
    text = re.sub(r'[^\w\s]', ' ', text)         # hapus tanda baca
    words = [SLANG_DICT.get(w, w) for w in text.split()]
    text = ' '.join(words)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def translate_batch(text_list: list, batch_size: int = 128) -> list:
    """Translate list teks dari Inggris ke Indonesia (NLLB-200)."""
    results = []
    tokenizer.src_lang = 'eng_Latn'
    bos_id = tokenizer.convert_tokens_to_ids('ind_Latn')

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=128
        ).to(device)

        with torch.no_grad():
            tokens = translate_model.generate(
                **inputs, forced_bos_token_id=bos_id,
                max_new_tokens=128, num_beams=1, do_sample=False
            )

        decoded = tokenizer.batch_decode(tokens, skip_special_tokens=True)
        results.extend(decoded)
        del inputs, tokens

    return results


def smart_preprocess(texts: list, translate_batch_size: int = 128) -> list:
    """
    Deteksi bahasa setiap teks:
    - Sudah Indonesia/Melayu → langsung dikembalikan
    - Bahasa lain (Inggris, dll.) → ditranslate ke Indonesia
    """
    results = [''] * len(texts)
    to_translate_idx, to_translate_txt = [], []

    for i, text in enumerate(texts):
        lang = detect_lang(text)
        if lang in SKIP_LANGS:
            results[i] = text
        else:
            to_translate_idx.append(i)
            to_translate_txt.append(text)

    n_id = len(texts) - len(to_translate_idx)
    n_en = len(to_translate_idx)
    pct  = n_id / len(texts) * 100 if texts else 0
    print(f'    Bahasa Indonesia: {n_id} ({pct:.0f}%) | Perlu translate: {n_en}')

    if to_translate_txt:
        translated = translate_batch(to_translate_txt, batch_size=translate_batch_size)
        for idx, val in zip(to_translate_idx, translated):
            results[idx] = val

    return results


print('Fungsi-fungsi siap.')

In [ ]:
# CELL 6: EKSEKUSI — PROSES SEMUA FOLDER
TRANSLATE_BATCH = 128  # T4 15GB — aman pakai batch besar

total_processed = 0

for folder in FOLDERS:
    raw_dir   = os.path.join(INPUT_ROOT, folder)
    clean_dir = os.path.join(OUTPUT_ROOT, folder)

    if not os.path.exists(raw_dir):
        print(f'[SKIP] Folder tidak ditemukan: {raw_dir}')
        continue

    csv_files = [f for f in os.listdir(raw_dir) if f.endswith('.csv')]
    print(f'\n=== {folder} ({len(csv_files)} file) ===')

    for idx, filename in enumerate(csv_files):
        print(f'  [{idx+1}/{len(csv_files)}] {filename}...')

        try:
            df = pd.read_csv(os.path.join(raw_dir, filename))

            # Filter baris kosong/tidak valid
            df = df.dropna(subset=['Review Text'])
            blacklist = {'N/A', 'n/a', 'na', 'nan', '-', '', ' ', 'null'}
            df = df[~df['Review Text'].astype(str).str.lower().str.strip().isin(blacklist)]
            df = df.drop(columns=[c for c in ['No', 'Review Count'] if c in df.columns], errors='ignore')

            # Clean teks
            df['Review Text Cleaned'] = df['Review Text'].apply(clean_text)
            df = df[df['Review Text Cleaned'].str.len() > 2].copy()

            if len(df) == 0:
                print('    Kosong setelah filter.')
                continue

            texts = df['Review Text Cleaned'].tolist()

            # Smart preprocess: deteksi bahasa + translate jika perlu
            processed = smart_preprocess(texts, translate_batch_size=TRANSLATE_BATCH)
            df['Review Text Cleaned'] = processed

            # Simpan
            out_name = filename.replace('.csv', '_Clean.csv')
            out_path = os.path.join(clean_dir, out_name)
            df = df.drop(columns=['Review Text'], errors='ignore')
            df = df.rename(columns={'Review Text Cleaned': 'Review Text'})
            df.to_csv(out_path, index=False, encoding='utf-8-sig')

            total_processed += len(df)
            print(f'    ✅ {len(df)} data → {out_path}')

        except Exception as e:
            print(f'    ❌ Error: {e}')

        # Cleanup
        try:
            del df, texts, processed
            gc.collect()
            torch.cuda.empty_cache()
        except Exception:
            pass

print(f'\n✅ SELESAI! Total {total_processed:,} review diproses.')
print(f'Output tersimpan di: {OUTPUT_ROOT}')

In [ ]:
# CELL 7: CEK HASIL
clean_files = glob.glob(os.path.join(OUTPUT_ROOT, '**', '*.csv'), recursive=True)
print(f'Total file clean: {len(clean_files)}')

# Tampilkan ukuran per folder
for folder in FOLDERS:
    folder_files = [f for f in clean_files if folder in f]
    total_rows = 0
    for f in folder_files:
        try:
            total_rows += len(pd.read_csv(f))
        except Exception:
            pass
    print(f'  {folder}: {len(folder_files)} file, {total_rows:,} baris')

print('\n📥 Download folder DatasetHotelCLEAN dari Output tab,')
print('   lalu upload sebagai Kaggle Dataset baru untuk Notebook 2.')